In [55]:
from pymongo import MongoClient
import re
import os
from typing import List, Dict, Any
import shutil

In [56]:
def connect_mongodb(uri,db_name, collection_name):
    # Connect to MongoDB 
    client = MongoClient(uri)
    
    # Select the database and collection
    db = client[db_name]
    collection = db[collection_name]
    
    return collection
def get_all_documents(collection):
    """
    Fetch all documents from the given MongoDB collection.
    Returns a list of dictionaries.
    """
    documents = list(collection.find())
    return documents
def get_keywords(data_dict):
    """
    Extracts and returns all keywords (keys) from the given dictionary.
    """
    return list(data_dict.keys())
def remove_element_case_insensitive(data_list, value_to_remove):
    """
    Remove all occurrences of value_to_remove from data_list, ignoring case.
    Returns the updated list.
    """
    return [item for item in data_list if item.lower() != value_to_remove.lower()]
def find_files_with_keyword(directory, keyword):
    matching_files = []
    keyword = keyword.lower()  
    for root, _, files in os.walk(directory):
        for file in files:
            file_path = os.path.join(root, file)
            try:
                with open(file_path, 'r', encoding='utf-8') as f:
                    content = f.read()
                    if keyword in content:
                        matching_files.append(file_path)
            except Exception as e:
                print(f"Error reading {file_path}: {e}")
    return matching_files
def get_all_directories(root_path):
    """
    Get all directory names under the given root path.
    Returns a list of directory names.
    """
    return [name for name in os.listdir(root_path) if os.path.isdir(os.path.join(root_path, name))]
def find_list_name(value,code_source_directory_standalone_name,code_source_directory_app_name):
    """
    Check which list the input value belongs to.
    Returns the name of the list or None if not found.
    """
    if value in code_source_directory_standalone_name:
        return code_source_directory_standalone,code_destination_directory_standalone
    elif value in code_source_directory_app_name:
        return code_source_directory_app,code_destination_directory_app
    else:
        return None, None
def copy_file_to_directory(source_file, dest_directory):
    """
    Copy a source file to the destination directory.
    Create the destination directory if it does not exist.
    """
    # Create destination directory if it doesn't exist
    if not os.path.exists(dest_directory):
        os.makedirs(dest_directory)

    # Get the filename from source path
    filename = os.path.basename(source_file)

    # Full destination file path
    dest_file_path = os.path.join(dest_directory, filename)

    # Copy the file
    shutil.copy2(source_file, dest_file_path)

    # print(f"Copied {source_file} to {dest_file_path}")

In [57]:
mongoDB_uri = 'mongodb://localhost:27017'
mongoDB_database = 'wearable-project-2' 
mongoDB_collection = 'dataset'
mongoDB_collection_2 = 'third_party_services'

In [58]:
code_source_directory_standalone = r'E:\wearable-project-decompile-source\source-code-standalone'
code_source_directory_app = r'E:\wearable-project-decompile-source\source-code-wearable'
code_destination_directory_standalone = r'E:\wearable-standalone-code-extraction'
code_destination_directory_app = r'E:\wearable-app-code-extraction'

In [59]:
code_source_directory_standalone_name = get_all_directories(code_source_directory_standalone)
code_source_directory_app_name = get_all_directories(code_source_directory_app)
# code_destination_directory_standalone_name = get_all_directories(code_destination_directory_standalone)
# code_destination_directory_app_name = get_all_directories(code_destination_directory_app)

In [60]:
print(len(code_source_directory_standalone_name))

73


In [61]:
print(len(code_source_directory_app_name))

5162


In [62]:
# Connect to the MongoDB collection
collection = connect_mongodb(mongoDB_uri,mongoDB_database,mongoDB_collection)
# Fetch the specified fields only
result_data = get_all_documents(collection)
print(len(result_data))

696


In [63]:
for i in range(len(result_data)):
    # i = 31
    data = result_data[i]
    app_id = data["_id"]
    app_third_party_keyword = data.get("third-party-keyword")
    # print("App third party keyword: ",app_third_party_keyword)
    if app_third_party_keyword!= None:
        print("---------------- Loop-" +str(i)+" ----------------")
        print("App id: ",app_id)
        source_directory, destination_directory = find_list_name(app_id+".apk",code_source_directory_standalone_name,code_source_directory_app_name)
        print("source_directory: ",source_directory)
        print("destination_directory: ",destination_directory)
        third_party_name_array = get_keywords(app_third_party_keyword)
        print("Third party name_array: ",third_party_name_array)
        search_third_party_array = remove_element_case_insensitive(third_party_name_array, "google")
        print("search_third_party_array: ",search_third_party_array)
        if(len(search_third_party_array)>0):
            for j in range(len(search_third_party_array)):
                search_third_party_keyword = (search_third_party_array[j]).strip()
                print("======Third party "+str(j)+": "+search_third_party_keyword)
                source_directory_search = source_directory+"\\"+app_id+".apk"
                print("source directory search: ",source_directory_search)
                search_result = find_files_with_keyword(source_directory_search, search_third_party_keyword)
                print("search_result: ",len(search_result))
                source_dest_directory = destination_directory+"\\"+app_id+"\\original\\"+search_third_party_keyword
                print("source dest directory: ",source_dest_directory)
                for x in range(len(search_result)):
                    file_name = search_result[x]
                    copy_file_to_directory(file_name, source_dest_directory)
                print("******Copy DONE for "+str(j)+": "+search_third_party_keyword)
    # break

---------------- Loop-0 ----------------
App id:  in.codeseed.audify
source_directory:  E:\wearable-project-decompile-source\source-code-wearable
destination_directory:  E:\wearable-app-code-extraction
Third party name_array:  ['Google']
search_third_party_array:  []
---------------- Loop-1 ----------------
App id:  com.urbandroid.lux
source_directory:  E:\wearable-project-decompile-source\source-code-wearable
destination_directory:  E:\wearable-app-code-extraction
Third party name_array:  ['Google']
search_third_party_array:  []
---------------- Loop-2 ----------------
App id:  maxcom.toolbox.altimeter
source_directory:  E:\wearable-project-decompile-source\source-code-wearable
destination_directory:  E:\wearable-app-code-extraction
Third party name_array:  ['Google']
search_third_party_array:  []
---------------- Loop-5 ----------------
App id:  com.mauriciotogneri.shoppinglist
source_directory:  E:\wearable-project-decompile-source\source-code-wearable
destination_directory:  E:\wea

Number of file contain keyword com.zendesk is: 3
Files containing the keyword com.zendesk
E:\wearable-project-decompile-source\source-code-standalone\app.groupcal.www-java\com\zendesk\service\ZendeskDateTypeAdapter.java
E:\wearable-project-decompile-source\source-code-standalone\app.groupcal.www-java\zendesk\belvedere\v.java
E:\wearable-project-decompile-source\source-code-standalone\app.groupcal.www-java\zendesk\core\ZendeskApplicationModule.java
